# Backtesting Strategy

Comprehensive backtesting with 8 financial metrics

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
from sklearn.preprocessing import RobustScaler
import tensorflow as tf

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

## Load LSTM Predictions

In [ ]:
# Load data
path = kagglehub.dataset_download('oussamataghlaoui/btc-oracle-on-chain-sentiment-and-macro-data')
import glob
csv_files = glob.glob(path + '/*.csv')
df = pd.read_csv(csv_files[0], parse_dates=['Datetime'], index_col='Datetime')
print(f'Data shape: {df.shape}')

## Financial Metrics Functions

In [ ]:
def calculate_sharpe_ratio(returns, risk_free_rate=0.02):
    excess = returns - risk_free_rate/252
    return np.sqrt(252) * excess.mean() / excess.std() if returns.std() > 0 else 0

def calculate_max_drawdown(returns):
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    return drawdown.min()

def calculate_sortino_ratio(returns, risk_free_rate=0.02):
    downside = returns[returns < 0]
    downside_std = downside.std()
    return np.sqrt(252) * returns.mean() / downside_std if downside_std > 0 else 0

def calculate_calmar_ratio(returns):
    annual_return = (1 + returns.mean()) ** 252 - 1
    max_dd = abs(calculate_max_drawdown(returns))
    return annual_return / max_dd if max_dd > 0 else 0

def calculate_metrics(returns, benchmark_returns):
    metrics = {}
    metrics['sharpe_ratio'] = calculate_sharpe_ratio(returns)
    metrics['max_drawdown'] = calculate_max_drawdown(returns)
    metrics['sortino_ratio'] = calculate_sortino_ratio(returns)
    metrics['calmar_ratio'] = calculate_calmar_ratio(returns)
    metrics['win_rate'] = (returns > 0).sum() / len(returns) if len(returns) > 0 else 0
    gross_profit = returns[returns > 0].sum()
    gross_loss = abs(returns[returns < 0].sum())
    metrics['profit_factor'] = gross_profit / gross_loss if gross_loss > 0 else np.inf
    # Beta & Alpha
    covariance = np.cov(returns, benchmark_returns)[0][1] if len(returns) == len(benchmark_returns) else 0
    benchmark_var = np.var(benchmark_returns)
    metrics['beta'] = covariance / benchmark_var if benchmark_var > 0 else 0
    metrics['alpha'] = returns.mean() - (0.02/252 + metrics['beta'] * (benchmark_returns.mean() - 0.02/252))
    return metrics

print('✅ Metrics functions defined')

## Results Summary

In [ ]:
print('Backtesting framework ready!')
print('Next: Load LSTM model and run backtest')